# Hunyuan3D-2mini Turbo — Google Colab Edition

**Image → 3D shape generation | GLB / OBJ / PLY / STL export**

Designed for a Google Colab GPU runtime, especially **T4 16GB**. This notebook uses Tencent's official Hunyuan3D-2 repository and its **0.6B mini Turbo shape model**.

### Before running
1. Open **Runtime → Change runtime type → T4 GPU** (if available).
2. Run the cells from top to bottom.
3. Upload one clear object image in the web interface.
4. Generate the 3D shape and download the exported model.

> **Important limitations:** This notebook runs the mini shape-generation model only. It does not load the much heavier texture-generation model, so the generated mesh may be untextured/white. This is primarily **image-to-3D**, not text-to-3D. Colab GPU availability and session limits are controlled by Google.
>
> **License:** Hunyuan3D is subject to Tencent's Hunyuan Community License (including non-commercial-use conditions). Read the license and confirm your intended use is allowed before using the model commercially.

Official repository: https://github.com/Tencent-Hunyuan/Hunyuan3D-2
Model: https://huggingface.co/tencent/Hunyuan3D-2mini


In [ ]:
# 1) Check the Colab runtime
import os, sys, subprocess, textwrap, time, pathlib

print("Python:", sys.version)
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    else:
        raise RuntimeError("GPU not detected. In Colab select Runtime > Change runtime type > T4 GPU, then rerun.")
except ImportError:
    raise RuntimeError("PyTorch is not available in this runtime. Reconnect to a Colab GPU runtime.")


## 2) Clone the official repository

This downloads the inference code. Model weights are downloaded separately when the app starts, so the first launch can take time.


In [ ]:
%cd /content
import os
import shutil
import subprocess
import time

REPO_DIR = "/content/Hunyuan3D-2"
MAIN_FILE = os.path.join(REPO_DIR, "gradio_app.py")

# Fail loudly instead of silently continuing with a broken/partial clone.
if os.path.isdir(REPO_DIR) and not os.path.isfile(MAIN_FILE):
    print("⚠️ Removing an incomplete Hunyuan3D-2 clone ...")
    shutil.rmtree(REPO_DIR)

if not os.path.isfile(MAIN_FILE):
    for attempt in range(1, 4):
        print(f"📥 Cloning official Hunyuan3D-2 repo (attempt {attempt}/3) ...")
        result = subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git", REPO_DIR],
            check=False,
        )
        if result.returncode == 0 and os.path.isfile(MAIN_FILE):
            break
        print("⚠️ Clone failed — retrying in 3 seconds ...")
        time.sleep(3)
    else:
        raise RuntimeError(
            "❌ Could not clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2. "
            "Check internet access, then restart the runtime and re-run this cell."
        )
else:
    print("✅ Hunyuan3D-2 already present — skipping clone.")

%cd /content/Hunyuan3D-2


## 3) Install dependencies

The official project instructions use `requirements.txt` followed by editable installation. This may take several minutes. If a dependency install changes PyTorch or CUDA packages, restart the runtime and rerun from the GPU check.


In [ ]:
%cd /content/Hunyuan3D-2
import subprocess
import sys

print("📥 Installing requirements.txt ...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=False,
)
if result.returncode != 0:
    raise RuntimeError("❌ pip install -r requirements.txt failed — see the error above.")

print("📥 Installing the package in editable mode ...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    check=False,
)
if result.returncode != 0:
    raise RuntimeError("❌ pip install -e . failed — see the error above.")

print("✅ Hunyuan3D-2 dependencies installed.")


## 4) Start the Hunyuan3D-2mini Turbo web interface

The notebook launches the official Gradio interface in **low-VRAM mode**, disables the texture stage to keep the workload suitable for a 16GB T4, and enables the Turbo/FlashVDM path.

If startup fails with a FlashVDM-related error, rerun this cell after removing `--enable_flashvdm` from the command. The standard mini Turbo model can still run, but may be slower.


In [ ]:
%cd /content/Hunyuan3D-2
import subprocess, time, os, pathlib, signal

# Bound to localhost only: the Colab port proxy below adds Google login.
# Stop a previous app process if this cell is rerun.
if "hunyuan_proc" in globals() and hunyuan_proc.poll() is None:
    hunyuan_proc.terminate()
    time.sleep(2)

log_path = "/content/hunyuan3d_app.log"
log_file = open(log_path, "w")

cmd = [
    sys.executable, "gradio_app.py",
    "--model_path", "tencent/Hunyuan3D-2mini",
    "--subfolder", "hunyuan3d-dit-v2-mini-turbo",
    "--low_vram_mode",
    "--disable_tex",
    "--enable_flashvdm",
    "--host", "127.0.0.1",
    "--port", "8080"
]

hunyuan_proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    cwd="/content/Hunyuan3D-2"
)

# Wait for the server to start or fail.
ready = False
for _ in range(180):
    if hunyuan_proc.poll() is not None:
        break
    try:
        import requests
        r = requests.get("http://127.0.0.1:8080", timeout=2)
        if r.status_code < 500:
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if ready:
    print("Hunyuan3D interface is running on port 8080.")
else:
    print("The app did not become ready. Recent log output:")
    print(pathlib.Path(log_path).read_text(errors="ignore")[-7000:])


## 5) Open the 3D generator

When the previous cell reports that the app is running, execute this cell to display the Gradio app inside Colab.


In [ ]:
#@title 🖥️ 5) Open the Gradio UI (inline iframe)
# `serve_kernel_port_as_window` prints https://localhost:8080/ which the local
# browser resolves to YOUR machine, not the Colab VM -> ERR_CONNECTION_REFUSED.
# Colab's own warning recommends the iframe, so use that instead.
import socket

# Guard: a dead app would show a blank/broken iframe with no explanation.
with socket.socket() as sock:
    if sock.connect_ex(("127.0.0.1", 8080)) != 0:
        raise RuntimeError(
            "❌ Hunyuan3D app is NOT running on port 8080.\n"
            "Re-run the START cell (cell 7) first. If you used 'Run all', do not "
            "run the STOP cell at the end."
        )
print("✅ App is listening on 127.0.0.1:8080 — opening Gradio UI below.")

from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(8080, height=950)


In [ ]:
#@title 🔗 Gradio ka apna public link (gradio.live) — EK CLICK MEIN
# Colab ke localhost/iframe/cloudflared tarike is browser me fail ho rahe the.
# Isliye official app ki aakhri line badal kar Gradio ke own share tunnel par chalao.
# Output me https://xxxx.gradio.live milega — kisi bhi browser me khul jayega.
import os
import re
import subprocess
import sys
import time

REPO = "/content/Hunyuan3D-2"
SRC = os.path.join(REPO, "gradio_app.py")
DST = os.path.join(REPO, "gradio_app_share.py")
LOG = "/tmp/hunyuan_share.log"

# Purana app band karo (GPU double-load se OOM na ho)
subprocess.run(["pkill", "-9", "-f", "gradio_app.py"], check=False)
time.sleep(2)

src = open(SRC, encoding="utf-8").read()
old_line = "    uvicorn.run(app, host=args.host, port=args.port, workers=1)"
if old_line not in src:
    raise RuntimeError("Official gradio_app.py badal gaya — ye trick ab kaam nahi karegi.")
open(DST, "w", encoding="utf-8").write(
    src.replace(old_line, "    demo.launch(share=True, debug=True)")
)

log = open(LOG, "w")
share_proc = subprocess.Popen(
    [sys.executable, DST,
     "--model_path", "tencent/Hunyuan3D-2mini",
     "--subfolder", "hunyuan3d-dit-v2-mini-turbo",
     "--low_vram_mode",
     "--disable_tex",
     "--enable_flashvdm"],
    cwd=REPO,
    stdout=log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(180):
    time.sleep(2)
    text = open(LOG, errors="replace").read()
    match = re.search(r"https://[a-zA-Z0-9\-]+\.gradio\.live", text)
    if match:
        public_url = match.group(0)
        break
    if share_proc.poll() is not None or "Traceback (most recent call last)" in text:
        break

if public_url:
    print("✅ Gradio UI ready!")
    print("🔗 Public link:", public_url)
    print("⚠️ Public link — sirf tum use karo, share mat karo.")
else:
    print("❌ Link nahi mila. Last log lines:")
    print(open(LOG, errors="replace").read()[-3000:])


## Usage tips

- Use a clear, well-lit image with one object and a simple background.
- A front/three-quarter view usually gives the model more useful shape information.
- If generation runs out of memory, restart the runtime and try again with no other GPU-heavy notebook sessions.
- The mini model generates geometry; do not expect a fully rigged, animation-ready character.
- Export the result as **GLB** for web/Three.js workflows, or OBJ/PLY/STL where supported.
- To stop the app, run the cell below.


In [ ]:
# Optional: stop the web app and release GPU memory.
# Default OFF so "Run all" does not kill the app you just started.
STOP_APP = False #@param {type:"boolean"}

if STOP_APP:
    if "hunyuan_proc" in globals() and hunyuan_proc.poll() is None:
        hunyuan_proc.terminate()
        print("Hunyuan3D app stopped.")
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
else:
    print("STOP_APP = False — app left running (safe with 'Run all').")
